In [42]:
import torch as t 
from torch import nn
import pandas as pd 
from pprint import pprint


df = pd.read_json("data.json")

x = df["input"]

tokenized = []
for sentence in x : 
    tokenized.append(sentence.split())
    
    
flatten = [word.lower() for sentence in tokenized for word in sentence]
for word in df['target']:
    flatten.append(word)


vocab = {word:idx for idx,word in enumerate(set(flatten))}
print (vocab)
# turn tokenized to ids
final_x = []
for sentence in tokenized: 
    hold = []
    for word in sentence : 
        if word in vocab.keys():
            hold.append(vocab[word])
    final_x.append(t.tensor(hold))


from torch.nn.utils.rnn import pad_sequence

vocab_size = len(vocab)
batch_input = pad_sequence(final_x , batch_first=True , padding_side="right")

batch_input

# target_indices لكل جملة
# target_indices = t.tensor([sample["target_index"] for sample in df])
indices = []
for index in df["target_index"]:
    indices.append(t.tensor(index))
    
    
target_words = t.tensor([vocab[sample] for sample in df['target']])



pprint((batch_input , indices ,target_words))


{'apple': 0, 'going': 1, 'sunday.': 2, 'market': 3, 'went': 4, 'sun': 5, 'the': 6, 'roof': 7, 'at': 8, 'before': 9, 'milk.': 10, 'farm.': 11, 'buy': 12, 'west': 13, 'a': 14, 'sat': 15, 'he': 16, 'cat': 17, 'night.': 18, 'read': 19, 'every': 20, 'day.': 21, 'sets': 22, 'book': 23, 'ali': 24, 'store': 25, 'yesterday': 26, 'play': 27, 'to': 28, 'on': 29, 'she': 30, 'last': 31, 'football': 32, 'bed.': 33, 'they': 34, 'in': 35, '[mask]': 36, 'ate': 37, 'an': 38, 'ahmed': 39, 'yesterday.': 40}
(tensor([[38,  0, 26,  8,  6, 11],
        [ 4, 28,  6, 40,  0,  0],
        [32, 20,  0,  0,  0,  0],
        [17, 29,  6,  7, 31, 18],
        [14, 23,  9,  1, 28, 33],
        [28,  6, 25, 28, 12, 10],
        [ 5, 35,  6, 13, 20, 21]]),
 [tensor(1), tensor(4), tensor(1), tensor(2), tensor(1), tensor(1), tensor(2)],
 tensor([37,  3, 27, 15, 19,  4, 22]))


In [43]:
class Model(nn.Module):
    def __init__(self , vocab_size , embedding_size , hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size , embedding_size)
        self.rnn = nn.LSTM(embedding_size , hidden_size , bidirectional=True)
        self.linear = nn.Linear(hidden_size*2 , vocab_size)
        
    def forward(self , x , indices):
        emb = self.embedding(x)
        out , _ = self.rnn(emb)
        mask_hidden = out[t.arange(x.size(0)),indices, : ]
        out = self.linear(mask_hidden)
        return out 

In [44]:

embedding_dim = 8
hidden_dim = 16
learning_rate = 0.01
num_epochs = 1000


model = Model(vocab_size=len(vocab), embedding_size=embedding_dim, hidden_size=hidden_dim)
optimizer = t.optim.Adam(model.parameters(), lr=learning_rate)

import torch.nn.functional as F

for epoch in range(1, num_epochs+1):
    model.train()
    optimizer.zero_grad()
    logits = model(batch_input, indices)
    loss = F.cross_entropy(logits, target_words)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


model.eval()
with t.no_grad():
    logits = model(batch_input, indices)
    predicted_ids = t.argmax(logits, dim=1)


id_to_word = {id_: word for word, id_ in vocab.items()}
predicted_words = [id_to_word[i.item()] for i in predicted_ids]

print("\nPredicted words:", predicted_words)
print("Target words:", [id_to_word[i.item()] for i in target_words])

Epoch 50, Loss: 0.0209
Epoch 100, Loss: 0.0050
Epoch 150, Loss: 0.0027
Epoch 200, Loss: 0.0018
Epoch 250, Loss: 0.0013
Epoch 300, Loss: 0.0009
Epoch 350, Loss: 0.0007
Epoch 400, Loss: 0.0006
Epoch 450, Loss: 0.0005
Epoch 500, Loss: 0.0004
Epoch 550, Loss: 0.0003
Epoch 600, Loss: 0.0003
Epoch 650, Loss: 0.0003
Epoch 700, Loss: 0.0002
Epoch 750, Loss: 0.0002
Epoch 800, Loss: 0.0002
Epoch 850, Loss: 0.0002
Epoch 900, Loss: 0.0001
Epoch 950, Loss: 0.0001
Epoch 1000, Loss: 0.0001

Predicted words: ['ate', 'market', 'play', 'sat', 'read', 'went', 'sets']
Target words: ['ate', 'market', 'play', 'sat', 'read', 'went', 'sets']


In [45]:
import torch as t

# جملة جديدة (توكنز + [MASK])
new_sentence = ["she" , "went" , "to" , "the" , "[MASK]" , "yesterday"]
new_target_index = 4  # مكان [MASK]


new_input_ids = t.tensor([[vocab[word.lower()] for word in new_sentence]], dtype=t.long)  

# Prediction
model.eval()
with t.no_grad():
    logits = model(new_input_ids, t.tensor([new_target_index]))
    probs = t.softmax(logits, dim=1)         
    predicted_id = t.argmax(probs, dim=1).item()

predicted_word = [word for word, id_ in vocab.items() if id_ == predicted_id][0]

print("Predicted word at [MASK]:", predicted_word)
print("Probabilities:", probs)

Predicted word at [MASK]: play
Probabilities: tensor([[0.0059, 0.0059, 0.0065, 0.0115, 0.0229, 0.0040, 0.0077, 0.0055, 0.0059,
         0.0074, 0.0060, 0.0058, 0.0069, 0.0060, 0.0056, 0.0225, 0.0058, 0.0075,
         0.0057, 0.0702, 0.0060, 0.0068, 0.0192, 0.0064, 0.0046, 0.0061, 0.0071,
         0.5990, 0.0086, 0.0067, 0.0055, 0.0068, 0.0055, 0.0073, 0.0068, 0.0072,
         0.0059, 0.0416, 0.0041, 0.0078, 0.0058]])
